# Analyze compaction zones

In [ ]:
from pathlib import Path
import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)

import bioio_ome_tiff

In [ ]:
input_dirpath = Path(input())

In [ ]:
ROI_names = [path.name for path in input_dirpath.glob('*.ome.tif')]
ROI_names.sort()

proc_dirpath = utils.get_proc_dirpath(input_dirpath)
analysis_df_path = proc_dirpath / dn.tables_dirname / f'cmp_zone_analysis.csv'

analysis_df = pd.DataFrame()

In [ ]:
def compute_cmp_zone_areas(cmp_seg, ROI_name, pixel_area):

    size_t = cmp_seg.shape[0]

    indiv_df = pd.DataFrame()
    
    for t in range(size_t):
        cmp_seg_t = cmp_seg[t, :, :, :, :]
        cmp_labelled, _ = ndi.label(cmp_seg_t)
        cmp_props = measure.regionprops_table(cmp_labelled.squeeze(),
                                  properties=['label', 'area'])
        indiv_df_t = pd.DataFrame(cmp_props)
        indiv_df_t = indiv_df_t.rename(columns={'area': 'area (pixels^2)'})
        indiv_df_t.insert(0, 'ROI imgname', ROI_name)
        indiv_df_t.insert(1, 't', t)             
        indiv_df = pd.concat([indiv_df, indiv_df_t])

    indiv_df['area (microns^2)'] = indiv_df['area (pixels^2)'] * pixel_area
    
    return indiv_df


In [ ]:
seg_cmpch = -3

utils.safe_save_csv(analysis_df, analysis_df_path)
for ROI_name in tqdm(ROI_names):

    # open image
    imgpath = input_dirpath / ROI_name
    img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
    img = img_file.data

    cmp_seg = (img[:, seg_cmpch, np.newaxis, :, :, :] > 0).astype('int')
    ROI_name = imgpath.name
    pixel_area = img_file.physical_pixel_sizes.X * img_file.physical_pixel_sizes.Y

    indiv_df = compute_cmp_zone_areas(cmp_seg, ROI_name, pixel_area)

    analysis_df = pd.concat([analysis_df, indiv_df])

    analysis_df.to_csv(analysis_df_path, index=False)


print('Done!')